In [ ]:
import re
import pandas as pd

In [ ]:
file=open('WhatsApp Chat with Ha👻 Ha😂 Ha😇.txt','r',encoding='utf-8')

In [ ]:
data=file.read()

In [ ]:
#print(data)

In [ ]:
pattern = r"^\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}\s(?:am|pm)\s-\s"

In [ ]:
msgs = re.split(pattern, data, flags=re.MULTILINE)[1:]
#print(msgs)

In [ ]:
date=re.findall(pattern,data,flags=re.M)
clean_dates = [d.replace('\u202f', ' ') for d in date]
#clean_dates

In [ ]:
df=pd.DataFrame({'user_message':msgs,'message_date':clean_dates})
df['message_date'] = df['message_date'].str.replace(' - ', '', regex=False).str.strip()
df['message_date']=pd.to_datetime(df['message_date'],format='%d/%m/%y, %I:%M %p')
df.rename(columns={'message_date':'date'},inplace=True)
df.head()

In [ ]:
df.shape

In [ ]:
users = []
messages = []
for message in df['user_message']:
    entry = re.split('^([\w\W]+?):\s', message, maxsplit=1)
    if len(entry) > 1:
        users.append(entry[1])
        messages.append(entry[2])
    else:
        users.append('group_notification')
        messages.append(entry[0])
df['user'] = users
df['message'] = messages
df.drop(columns=['user_message'], inplace=True)
df.head()

In [ ]:
df['year']=df['date'].dt.year

In [ ]:
df['month']=df['date'].dt.month_name()

In [ ]:
df['day']=df['date'].dt.day
df.head()

In [ ]:
df['hour']=df['date'].dt.hour


In [ ]:
df['minute']=df['date'].dt.minute

In [ ]:
df.head()

In [ ]:
 df[df['user']=='Mr. Shiii🥰👻'].shape

In [ ]:
words=[]
for msg in df['message']:
    words.extend(msg.split())

In [ ]:
len(words)

In [ ]:
pip install urlextract

In [ ]:
from urlextract import  URLExtract
ext=URLExtract()
urls=ext.find_urls('lets have url stackoverflow.com as an example google.com , http://facebook.com')
urls

In [ ]:
links=[]
for msg in df['message']:
    links.extend(ext.find_urls(msg))

In [ ]:
len(links)

In [ ]:
x=df['user'].value_counts().head()

In [ ]:
import matplotlib.pyplot as plt
name=x.index
count=x.values
plt.figure(figsize=(4,3))
plt.bar(name,count)
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
round((df['user'].value_counts()/df.shape[0])*100,2).reset_index()

In [ ]:
temp=df[df['user']!='group_notification']
temp=temp[temp['message']!='<Media omitted>\n']

In [ ]:
pip install telugu-stopwords

In [ ]:
from telugu_stopwords import tsw, rmtsw
telugu_sw = tsw()
stopwords = telugu_sw.get_stopwords(script="english")
# text='nenu ee roju chala happy ga unnanu'
# cleaned = rmtsw(text, script="english")
# print(cleaned)


In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
all_stopwords = set(stopwords).union(set(ENGLISH_STOP_WORDS))

In [ ]:
import string
all_stopwords.update(['message', 'omitted', 'media', 'haa', 'kooda', 'aithe', 'ade', 'lo'])
words=[]
for msg in temp['message']:
    for word in msg.lower().split():
        clean_word = word.strip(string.punctuation)
        if clean_word and clean_word not in all_stopwords and len(clean_word) > 1:
            words.append(clean_word)
    words.extend(msg.split())

In [ ]:
word=[]
for msg in df['message']:
    word.extend(msg.split())

In [ ]:
from collections import Counter
pd.DataFrame(Counter(word).most_common(20))

In [ ]:
pip install emoji

In [ ]:
import emoji
emojis=[]
for msg in df['message']:
    emojis.extend([c for c in msg if emoji.is_emoji(c)])

In [ ]:
pd.DataFrame(Counter(emojis).most_common(len(Counter(emojis))))

In [ ]:
df['month_num']=df['date'].dt.month
df.head()

In [ ]:
timeline=df.groupby(['year','month_num','month']).count()['message'].reset_index()

In [ ]:
time=[]
for i in range(timeline.shape[0]):
    time.append(timeline['month'][i] + "-" + str(timeline['year'][i]))

In [ ]:
timeline['time']=time
timeline

In [ ]:
plt.plot(timeline['time'],timeline['message'])
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
df['only_date']=df['date'].dt.date
daily_timeline=df.groupby('only_date').count()['message'].reset_index()

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(daily_timeline['only_date'],daily_timeline['message'])
plt.xticks(rotation='vertical')

In [ ]:
df['day_name']=df['date'].dt.day_name()
df.head()

In [ ]:
df['day_name'].value_counts()

In [ ]:
df['month'].value_counts()

In [ ]:
df.head()

In [ ]:
period = []

for hour in df['date'].dt.hour:
    start_suffix = "AM" if hour < 12 else "PM"
    start_h = hour % 12
    if start_h == 0: start_h = 12
    next_hour_24 = (hour + 1) % 24
    end_suffix = "AM" if next_hour_24 < 12 else "PM"
    end_h = next_hour_24 % 12
    if end_h == 0: end_h = 12
    period.append(f"{start_h} {start_suffix}-{end_h} {end_suffix}")

In [ ]:
df['period']=period

In [ ]:
df.sample(5)

In [ ]:
import seaborn as sns
plt.figure(figsize=(10,6))
sns.heatmap(df.pivot_table(index='day_name',columns='period',values='message',aggfunc='count').fillna(0))
plt.yticks(rotation='horizontal')
plt.show()